In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch import nn, optim
import random
from PIL import Image
import timm
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torchvision.models as models
import random

In [2]:
def set_random_seed(seed=42):
    os.environ["PYTHONHASHSEED"] = "42"
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) 
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    torch.use_deterministic_algorithms(True)

set_random_seed()

In [3]:
df = pd.read_csv('/kaggle/input/peta-train-test/PETA_train_list.txt', sep=" ", header=None).iloc[:,:36]
df[0] = '/kaggle/input/peta-v1/PETA dataset/'+df[0].str.split('/').str[1:].str.join("/")

train_set, val_set = train_test_split(df, test_size=0.1666, random_state=42, shuffle=True)

test_set = pd.read_csv('/kaggle/input/peta-train-test/PETA_test_list.txt', sep=" ", header=None).iloc[:,:36]
test_set[0] = '/kaggle/input/peta-v1/PETA dataset/'+test_set[0].str.split('/').str[1:].str.join("/")

In [4]:
class CustomImageDataset(Dataset):
    def __init__(self, df, transform=None):
        self.imgs = df.iloc[:,0].values
        self.labels = df.iloc[:,1:].values.astype('float32')
        self.transform = transform
        
    def __len__(self):
        return len(self.imgs)
    
    def __getitem__(self, idx):
        image = Image.open(self.imgs[idx]).convert('RGB')
        label = torch.from_numpy(self.labels[idx])
        
        if self.transform:
            image = self.transform(image)
        
        return image, label

In [5]:
train_transform = transforms.Compose([
    transforms.Resize((384, 192)),  # maintain pedestrian shape, taller than wide
    transforms.RandomHorizontalFlip(),  # common for pedestrian symmetry
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.Resize((384, 192), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor()
])

train_dataset = CustomImageDataset(train_set, train_transform)
val_dataset = CustomImageDataset(val_set, val_transform)
test_dataset = CustomImageDataset(test_set, val_transform)

In [6]:
r = train_set.iloc[:,1:].values.mean(0)
r = torch.from_numpy(r).type(torch.float32)

In [7]:
# Common DataLoader kwargs
loader_kwargs = {
    'batch_size': 16,
    'num_workers': 4,
    'pin_memory': True,
    'persistent_workers': True,
}

train_loader = DataLoader(
    train_dataset, 
    shuffle=True,
    **loader_kwargs
)

val_loader = DataLoader(
    val_dataset, 
    shuffle=False,
    **loader_kwargs
)

test_loader = DataLoader(
    test_dataset, 
    shuffle=False,
    **loader_kwargs
)

In [8]:
def compute_mFive(y_true, y_pred):
    y_true = np.array(y_true, dtype='float32')
    y_pred = np.array(y_pred, dtype='float32')
    
    # ---- mA (mean per-label accuracy) ----
    L = y_true.shape[1]
    TP = np.sum((y_true == 1) & (y_pred == 1), axis=0)
    TN = np.sum((y_true == 0) & (y_pred == 0), axis=0)
    P = np.sum(y_true == 1, axis=0)
    N = np.sum(y_true == 0, axis=0)

    tp_over_p = np.where(P == 0, 1.0, TP / P)
    tn_over_n = np.where(N == 0, 1.0, TN / N)

    mA = (1 / (2 * L)) * np.sum(tp_over_p + tn_over_n)

    # ---- Instance-level metrics ----
    intersection = np.logical_and(y_true, y_pred).sum(axis=1)
    union = np.logical_or(y_true, y_pred).sum(axis=1)
    pred_sum = y_pred.sum(axis=1)
    true_sum = y_true.sum(axis=1)

    eps = 1e-8  # small number to avoid division by zero
    Accuracy = np.mean(intersection / (union + eps))
    Precision = np.mean(intersection / (pred_sum + eps))
    Recall = np.mean(intersection / (true_sum + eps))
    F1 = 2 * Precision * Recall / (Precision + Recall)

    # ---- mFive ----
    mFive = np.mean([mA, Accuracy, Precision, Recall, F1])

    return {
        'mA': mA,
        'Accuracy': Accuracy,
        'Precision': Precision,
        'Recall': Recall,
        'F1': F1,
        'mFive': mFive
    }

In [9]:
def train_one_epoch(model, dataloader, criterion, optimizer, scheduler, device, epoch, num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} - Training", leave=False)
    for i, (images, labels) in enumerate(pbar, 1):
        images = images.to(device)
        labels = torch.abs(labels-torch.rand(labels.shape) * 0.2)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        preds = (torch.sigmoid(outputs) >= 0.5).float()
        labels = (labels >= 0.5).float()
        correct += (preds == labels).float().sum().item()
        total += labels.numel()

        pbar.set_postfix({
            'Avg Loss': f'{running_loss / i:.4f}',
            'Avg Acc': f'{correct / total:.4f}'
        })
        
    scheduler.step()
    return running_loss / len(dataloader), correct / total


def validate_one_epoch(model, dataloader, criterion, device, epoch, num_epochs):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{num_epochs} - Validation", leave=False)
    with torch.no_grad():
        for i, (images, labels) in enumerate(pbar, 1):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).float().sum().item()
            total += labels.numel()

            all_predictions.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                'Avg Loss': f'{running_loss / i:.4f}',
                'Avg Acc': f'{correct / total:.4f}'
            })

    metrics = compute_mFive(all_labels, all_predictions)
    return running_loss / len(dataloader), correct / total, metrics


def train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=20):
    model.to(device)
    best_val_mfive = 0
    train_losses, val_losses = [], []

    for epoch in range(num_epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, device, epoch, num_epochs)
        val_loss, val_acc, metrics = validate_one_epoch(model, val_loader, criterion, device, epoch, num_epochs)

        train_losses.append(train_loss)
        val_losses.append(val_loss)

        print(f"\nEpoch {epoch+1}/{num_epochs} Summary:")
        print(f"  Train Loss:     {train_loss:.4f} - Acc: {train_acc:.4f}")
        print(f"  Val Loss:{val_loss:.4f} - Val Acc: {val_acc:.4f}")
        print(f"  Val mFive:      {metrics['mFive']:.4f}")
        print(f"  Val Acc:         {metrics['Accuracy']:.4f}")
        print(f"  Val mA:         {metrics['mA']:.4f}")
        print(f"  Val F1:         {metrics['F1']:.4f}")
        print(f"  Val Precision:  {metrics['Precision']:.4f}")
        print(f"  Val Recall:     {metrics['Recall']:.4f}")
        print("-" * 60)

        if metrics["mFive"] > best_val_mfive:
            best_val_mfive = metrics["mFive"]
            torch.save(model.state_dict(), 'best_weight.pth')
            print(f"✅ Best model saved (mFive: {best_val_mfive:.4f})")

    return model, train_losses, val_losses

def test_model(model, test_loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_predictions = []
    all_labels = []

    pbar = tqdm(test_loader, desc="Testing", leave=False)
    with torch.no_grad():
        for i, (images, labels) in enumerate(pbar, 1):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()

            probs = torch.sigmoid(outputs)
            preds = (probs >= 0.5).float()

            correct += (preds == labels).float().sum().item()
            total += labels.numel()

            all_predictions.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            avg_loss = running_loss / i
            avg_acc = correct / total

            pbar.set_postfix({
                'Avg Loss': f'{avg_loss:.4f}',
                'Avg Acc': f'{avg_acc:.4f}'
            })

    metrics = compute_mFive(all_labels, all_predictions)

    print(f"\nTest Summary:")
    print(f"  Test Loss:     {running_loss / len(test_loader):.4f}")
    print(f"  Test mFive:    {metrics['mFive']:.4f}")
    print(f"  Test Accuracy: {metrics['Accuracy']:.4f}")
    print(f"  Test mA:       {metrics['mA']:.4f}")
    print(f"  Test F1:       {metrics['F1']:.4f}")
    print(f"  Test Precision:{metrics['Precision']:.4f}")
    print(f"  Test Recall:   {metrics['Recall']:.4f}")
    print("-" * 60)

    return metrics

In [10]:
class Eff_Baseline(nn.Module):
    def __init__(self, num_classes):
        super(Eff_Baseline, self).__init__()
        backbone = models.efficientnet_v2_m(weights="DEFAULT")
        self.backbone = backbone.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Linear(1280, num_classes)

    def forward(self, x):
        x = self.backbone(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [11]:
class WeightedBCELoss(nn.Module):
    
    def __init__(self, r):
        super(WeightedBCELoss, self).__init__()
        self.r = r 

    def forward(self, logits, targets): 
        probs = torch.sigmoid(logits) 
        pos_weights = torch.exp(1 - self.r).expand_as(targets)
        neg_weights = torch.exp(self.r).expand_as(targets)
        weighted_bce = pos_weights * targets * torch.log(probs + 1e-7) + neg_weights * (1 - targets) * torch.log(1 - probs + 1e-7)
        return (-weighted_bce).mean()


In [12]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model=Eff_Baseline(num_classes=35)
# Load checkpoint
checkpoint = torch.load('/kaggle/input/best-bb1/best_weight.pth', map_location=device)
# Remove only head weights
state_dict = {k: v for k, v in checkpoint.items() if not (k.startswith("classifier"))}
model.load_state_dict(state_dict, strict=False)

Downloading: "https://download.pytorch.org/models/efficientnet_v2_m-dc08266a.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_v2_m-dc08266a.pth
100%|██████████| 208M/208M [00:02<00:00, 78.9MB/s]


_IncompatibleKeys(missing_keys=['classifier.weight', 'classifier.bias'], unexpected_keys=[])

In [13]:
criterion = WeightedBCELoss(r.to(device)).to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=0.001)
    
# Learning rate scheduler
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=11, gamma=0.1)

In [14]:
model, train_losses, val_losses = train_model(model, train_loader, val_loader, criterion, optimizer, scheduler, device, num_epochs=50)


Epoch 1/50 Summary:
  Train Loss:     0.8194 - Acc: 0.8639
  Val Loss:0.5022 - Val Acc: 0.9026
  Val mFive:      0.7810
  Val Acc:         0.6948
  Val mA:         0.7927
  Val F1:         0.8056
  Val Precision:  0.7869
  Val Recall:     0.8253
------------------------------------------------------------
✅ Best model saved (mFive: 0.7810)



Epoch 2/50 Summary:
  Train Loss:     0.7381 - Acc: 0.9175
  Val Loss:0.4446 - Val Acc: 0.9190
  Val mFive:      0.8174
  Val Acc:         0.7461
  Val mA:         0.8265
  Val F1:         0.8379
  Val Precision:  0.8223
  Val Recall:     0.8542
------------------------------------------------------------
✅ Best model saved (mFive: 0.8174)



Epoch 3/50 Summary:
  Train Loss:     0.7015 - Acc: 0.9403
  Val Loss:0.4316 - Val Acc: 0.9224
  Val mFive:      0.8260
  Val Acc:         0.7571
  Val mA:         0.8370
  Val F1:         0.8451
  Val Precision:  0.8298
  Val Recall:     0.8611
------------------------------------------------------------
✅ Best model saved (mFive: 0.8260)



Epoch 4/50 Summary:
  Train Loss:     0.6757 - Acc: 0.9557
  Val Loss:0.4250 - Val Acc: 0.9256
  Val mFive:      0.8330
  Val Acc:         0.7678
  Val mA:         0.8443
  Val F1:         0.8509
  Val Precision:  0.8394
  Val Recall:     0.8627
------------------------------------------------------------
✅ Best model saved (mFive: 0.8330)



Epoch 5/50 Summary:
  Train Loss:     0.6579 - Acc: 0.9660
  Val Loss:0.4168 - Val Acc: 0.9255
  Val mFive:      0.8360
  Val Acc:         0.7702
  Val mA:         0.8527
  Val F1:         0.8522
  Val Precision:  0.8374
  Val Recall:     0.8676
------------------------------------------------------------
✅ Best model saved (mFive: 0.8360)


...


Epoch 46/50 Summary:
  Train Loss:     0.5921 - Acc: 0.9989
  Val Loss:0.3992 - Val Acc: 0.9373
  Val mFive:      0.8572
  Val Acc:         0.8005
  Val mA:         0.8685
  Val F1:         0.8723
  Val Precision:  0.8671
  Val Recall:     0.8775
------------------------------------------------------------



Epoch 47/50 Summary:
  Train Loss:     0.5923 - Acc: 0.9990
  Val Loss:0.3997 - Val Acc: 0.9372
  Val mFive:      0.8566
  Val Acc:         0.8002
  Val mA:         0.8675
  Val F1:         0.8717
  Val Precision:  0.8676
  Val Recall:     0.8759
------------------------------------------------------------



Epoch 48/50 Summary:
  Train Loss:     0.5925 - Acc: 0.9990
  Val Loss:0.3991 - Val Acc: 0.9372
  Val mFive:      0.8566
  Val Acc:         0.8000
  Val mA:         0.8679
  Val F1:         0.8717
  Val Precision:  0.8676
  Val Recall:     0.8759
------------------------------------------------------------



Epoch 49/50 Summary:
  Train Loss:     0.5924 - Acc: 0.9989
  Val Loss:0.3998 - Val Acc: 0.9370
  Val mFive:      0.8565
  Val Acc:         0.8000
  Val mA:         0.8668
  Val F1:         0.8719
  Val Precision:  0.8666
  Val Recall:     0.8773
------------------------------------------------------------



Epoch 50/50 Summary:
  Train Loss:     0.5923 - Acc: 0.9989
  Val Loss:0.3993 - Val Acc: 0.9371
  Val mFive:      0.8565
  Val Acc:         0.7999
  Val mA:         0.8672
  Val F1:         0.8718
  Val Precision:  0.8669
  Val Recall:     0.8767
------------------------------------------------------------


In [15]:
model.load_state_dict(torch.load('best_weight.pth'))
metrics = test_model(model, test_loader, criterion, device)


Test Summary:
  Test Loss:     0.3836
  Test mFive:    0.8648
  Test Accuracy: 0.8112
  Test mA:       0.8726
  Test F1:       0.8800
  Test Precision:0.8760
  Test Recall:   0.8839
------------------------------------------------------------
